# Counting what a hand-annotated text corpus holds

**Data kit lab** `biored-corpus-statistics` · Discovery · about 40 minutes

**The question.** How many gene, disease, chemical, variant, species and cell line mentions, how many unique concept identifiers and how many relations does the BioRED corpus hold in its training, development and test sets, and how are they stored?

**The data.** BioRED is a set of 600 PubMed abstracts. Annotators marked every span that names a gene, disease, chemical, variant, species or cell line, linked each span to an identifier in a public vocabulary, and recorded the relations between concepts in each abstract. NCBI publishes it at https://ftp.ncbi.nlm.nih.gov/pub/lu/BioRED/ under a public domain notice. Please cite the authors: Luo L, Lai P-T, Wei C-H, Arighi CN, Lu Z. BioRED: a rich biomedical relation extraction dataset. *Briefings in Bioinformatics* 2022;23(5):bbac282, https://doi.org/10.1093/bib/bbac282.

**The published result.** Table 4 of that paper counts mentions and unique identifiers by entity type and split, and relations by split. You rebuild it from the files.

**What you do.**

1. Read the notice and list the files in the zip.
2. Learn the PubTator format and parse it into three tables.
3. Map the type labels to the paper's six classes.
4. Rebuild Table 4: mentions, unique identifiers, relations, novel relations.
5. Count relation types and concept pairs.
6. Look at the choices that change a count.

**How a count is defined here** (fixed before the files were opened; see SPEC.txt in the lab folder):

- a **mention** is one annotated span, one line in the file, even if it carries several identifiers;
- a **unique concept identifier** is one distinct identifier within one class, after splitting a composite field such as `4312,4313` at the comma; `-` is not an identifier;
- a **relation** is one relation line in one document; the same pair in two abstracts counts twice.

**Dates.** The zip is dated 2022-04-07 in the folder listing, before the paper appeared (July 2022). If NCBI replaces the file, the first cell will say the download differs from the copy this lab was tested with. A count that then differs from the paper tells you the file changed, not that your code is wrong; compare the file date with the paper's date before you decide.

In [ ]:
# datakit:setup (generated from lab.yaml and expected.json by scripts/datakit/harness.py sync; do not edit by hand)
import hashlib, json, os, pathlib, urllib.request

LAB = 'biored-corpus-statistics'
FILES = json.loads(r'''{
 "corpus": {
  "url": "https://ftp.ncbi.nlm.nih.gov/pub/lu/BioRED/BIORED.zip",
  "name": "corpus.zip",
  "sha256": "c3032230bd89d22a0923d0df6ae943bc8ea37fba7e42dafa7a8dec21bac02d47",
  "bytes": 2191684
 },
 "readme": {
  "url": "https://ftp.ncbi.nlm.nih.gov/pub/lu/BioRED/README.txt",
  "name": "readme.txt",
  "sha256": "ceffff95c61b4c47319b50969052211ab0976c73e8cbf816e23a93ea1173e8e7",
  "bytes": 2377
 }
}''')
PUBLISHED = json.loads(r'''{
 "docs_train": {
  "label": "Documents, training",
  "value": 400,
  "tolerance": {
   "exact": true
  }
 },
 "docs_dev": {
  "label": "Documents, dev",
  "value": 100,
  "tolerance": {
   "exact": true
  }
 },
 "docs_test": {
  "label": "Documents, tests",
  "value": 100,
  "tolerance": {
   "exact": true
  }
 },
 "docs_total": {
  "label": "Documents, total",
  "value": 600,
  "tolerance": {
   "exact": true
  }
 },
 "mentions_all_train": {
  "label": "Entity mentions, All, training",
  "value": 13351,
  "tolerance": {
   "exact": true
  }
 },
 "ids_all_train": {
  "label": "Unique concept identifiers, All, training",
  "value": 2708,
  "tolerance": {
   "exact": true
  }
 },
 "mentions_all_dev": {
  "label": "Entity mentions, All, dev",
  "value": 3533,
  "tolerance": {
   "exact": true
  }
 },
 "ids_all_dev": {
  "label": "Unique concept identifiers, All, dev",
  "value": 956,
  "tolerance": {
   "exact": true
  }
 },
 "mentions_all_test": {
  "label": "Entity mentions, All, tests",
  "value": 3535,
  "tolerance": {
   "exact": true
  }
 },
 "ids_all_test": {
  "label": "Unique concept identifiers, All, tests",
  "value": 982,
  "tolerance": {
   "exact": true
  }
 },
 "mentions_all_total": {
  "label": "Entity mentions, All, total",
  "value": 20419,
  "tolerance": {
   "exact": true
  }
 },
 "ids_all_total": {
  "label": "Unique concept identifiers, All, total",
  "value": 3869,
  "tolerance": {
   "exact": true
  }
 },
 "mentions_gene_train": {
  "label": "Entity mentions, Gene, training",
  "value": 4430,
  "tolerance": {
   "exact": true
  }
 },
 "ids_gene_train": {
  "label": "Unique concept identifiers, Gene, training",
  "value": 1141,
  "tolerance": {
   "exact": true
  }
 },
 "mentions_gene_dev": {
  "label": "Entity mentions, Gene, dev",
  "value": 1087,
  "tolerance": {
   "exact": true
  }
 },
 "ids_gene_dev": {
  "label": "Unique concept identifiers, Gene, dev",
  "value": 368,
  "tolerance": {
   "exact": true
  }
 },
 "mentions_gene_test": {
  "label": "Entity mentions, Gene, tests",
  "value": 1180,
  "tolerance": {
   "exact": true
  }
 },
 "ids_gene_test": {
  "label": "Unique concept identifiers, Gene, tests",
  "value": 399,
  "tolerance": {
   "exact": true
  }
 },
 "mentions_gene_total": {
  "label": "Entity mentions, Gene, total",
  "value": 6697,
  "tolerance": {
   "exact": true
  }
 },
 "ids_gene_total": {
  "label": "Unique concept identifiers, Gene, total",
  "value": 1643,
  "tolerance": {
   "exact": true
  }
 },
 "mentions_disease_train": {
  "label": "Entity mentions, Disease, training",
  "value": 3646,
  "tolerance": {
   "exact": true
  }
 },
 "ids_disease_train": {
  "label": "Unique concept identifiers, Disease, training",
  "value": 576,
  "tolerance": {
   "exact": true
  }
 },
 "mentions_disease_dev": {
  "label": "Entity mentions, Disease, dev",
  "value": 982,
  "tolerance": {
   "exact": true
  }
 },
 "ids_disease_dev": {
  "label": "Unique concept identifiers, Disease, dev",
  "value": 244,
  "tolerance": {
   "exact": true
  }
 },
 "mentions_disease_test": {
  "label": "Entity mentions, Disease, tests",
  "value": 917,
  "tolerance": {
   "exact": true
  }
 },
 "ids_disease_test": {
  "label": "Unique concept identifiers, Disease, tests",
  "value": 244,
  "tolerance": {
   "exact": true
  }
 },
 "mentions_disease_total": {
  "label": "Entity mentions, Disease, total",
  "value": 5545,
  "tolerance": {
   "exact": true
  }
 },
 "ids_disease_total": {
  "label": "Unique concept identifiers, Disease, total",
  "value": 778,
  "tolerance": {
   "exact": true
  }
 },
 "mentions_chemical_train": {
  "label": "Entity mentions, Chemical, training",
  "value": 2853,
  "tolerance": {
   "exact": true
  }
 },
 "ids_chemical_train": {
  "label": "Unique concept identifiers, Chemical, training",
  "value": 486,
  "tolerance": {
   "exact": true
  }
 },
 "mentions_chemical_dev": {
  "label": "Entity mentions, Chemical, dev",
  "value": 822,
  "tolerance": {
   "exact": true
  }
 },
 "ids_chemical_dev": {
  "label": "Unique concept identifiers, Chemical, dev",
  "value": 184,
  "tolerance": {
   "exact": true
  }
 },
 "mentions_chemical_test": {
  "label": "Entity mentions, Chemical, tests",
  "value": 754,
  "tolerance": {
   "exact": true
  }
 },
 "ids_chemical_test": {
  "label": "Unique concept identifiers, Chemical, tests",
  "value": 170,
  "tolerance": {
   "exact": true
  }
 },
 "mentions_chemical_total": {
  "label": "Entity mentions, Chemical, total",
  "value": 4429,
  "tolerance": {
   "exact": true
  }
 },
 "ids_chemical_total": {
  "label": "Unique concept identifiers, Chemical, total",
  "value": 651,
  "tolerance": {
   "exact": true
  }
 },
 "mentions_variant_train": {
  "label": "Entity mentions, Variant, training",
  "value": 890,
  "tolerance": {
   "exact": true
  }
 },
 "ids_variant_train": {
  "label": "Unique concept identifiers, Variant, training",
  "value": 420,
  "tolerance": {
   "exact": true
  }
 },
 "mentions_variant_dev": {
  "label": "Entity mentions, Variant, dev",
  "value": 250,
  "tolerance": {
   "exact": true
  }
 },
 "ids_variant_dev": {
  "label": "Unique concept identifiers, Variant, dev",
  "value": 135,
  "tolerance": {
   "exact": true
  }
 },
 "mentions_variant_test": {
  "label": "Entity mentions, Variant, tests",
  "value": 241,
  "tolerance": {
   "exact": true
  }
 },
 "ids_variant_test": {
  "label": "Unique concept identifiers, Variant, tests",
  "value": 137,
  "tolerance": {
   "exact": true
  }
 },
 "mentions_variant_total": {
  "label": "Entity mentions, Variant, total",
  "value": 1381,
  "tolerance": {
   "exact": true
  }
 },
 "ids_variant_total": {
  "label": "Unique concept identifiers, Variant, total",
  "value": 678,
  "tolerance": {
   "exact": true
  }
 },
 "mentions_species_train": {
  "label": "Entity mentions, Species, training",
  "value": 1429,
  "tolerance": {
   "exact": true
  }
 },
 "ids_species_train": {
  "label": "Unique concept identifiers, Species, training",
  "value": 37,
  "tolerance": {
   "exact": true
  }
 },
 "mentions_species_dev": {
  "label": "Entity mentions, Species, dev",
  "value": 370,
  "tolerance": {
   "exact": true
  }
 },
 "ids_species_dev": {
  "label": "Unique concept identifiers, Species, dev",
  "value": 13,
  "tolerance": {
   "exact": true
  }
 },
 "mentions_species_test": {
  "label": "Entity mentions, Species, tests",
  "value": 393,
  "tolerance": {
   "exact": true
  }
 },
 "ids_species_test": {
  "label": "Unique concept identifiers, Species, tests",
  "value": 11,
  "tolerance": {
   "exact": true
  }
 },
 "mentions_species_total": {
  "label": "Entity mentions, Species, total",
  "value": 2192,
  "tolerance": {
   "exact": true
  }
 },
 "ids_species_total": {
  "label": "Unique concept identifiers, Species, total",
  "value": 47,
  "tolerance": {
   "exact": true
  }
 },
 "mentions_cellline_train": {
  "label": "Entity mentions, Cell Line, training",
  "value": 103,
  "tolerance": {
   "exact": true
  }
 },
 "ids_cellline_train": {
  "label": "Unique concept identifiers, Cell Line, training",
  "value": 48,
  "tolerance": {
   "exact": true
  }
 },
 "mentions_cellline_dev": {
  "label": "Entity mentions, Cell Line, dev",
  "value": 22,
  "tolerance": {
   "exact": true
  }
 },
 "ids_cellline_dev": {
  "label": "Unique concept identifiers, Cell Line, dev",
  "value": 12,
  "tolerance": {
   "exact": true
  }
 },
 "mentions_cellline_test": {
  "label": "Entity mentions, Cell Line, tests",
  "value": 50,
  "tolerance": {
   "exact": true
  }
 },
 "ids_cellline_test": {
  "label": "Unique concept identifiers, Cell Line, tests",
  "value": 21,
  "tolerance": {
   "exact": true
  }
 },
 "mentions_cellline_total": {
  "label": "Entity mentions, Cell Line, total",
  "value": 175,
  "tolerance": {
   "exact": true
  }
 },
 "ids_cellline_total": {
  "label": "Unique concept identifiers, Cell Line, total",
  "value": 72,
  "tolerance": {
   "exact": true
  }
 },
 "relations_train": {
  "label": "Relation, training",
  "value": 4178,
  "tolerance": {
   "exact": true
  }
 },
 "relations_dev": {
  "label": "Relation, dev",
  "value": 1162,
  "tolerance": {
   "exact": true
  }
 },
 "relations_test": {
  "label": "Relation, tests",
  "value": 1163,
  "tolerance": {
   "exact": true
  }
 },
 "relations_total": {
  "label": "Relation, total",
  "value": 6503,
  "tolerance": {
   "exact": true
  }
 },
 "novel_train": {
  "label": "Relation pair with novelty findings, training",
  "value": 2838,
  "tolerance": {
   "exact": true
  }
 },
 "novel_dev": {
  "label": "Relation pair with novelty findings, dev",
  "value": 835,
  "tolerance": {
   "exact": true
  }
 },
 "novel_test": {
  "label": "Relation pair with novelty findings, tests",
  "value": 859,
  "tolerance": {
   "exact": true
  }
 },
 "novel_total": {
  "label": "Relation pair with novelty findings, total",
  "value": 4532,
  "tolerance": {
   "exact": true
  }
 },
 "share_association": {
  "label": "Share of relations of type Association",
  "value": 52.0,
  "tolerance": {
   "abs": 0.5
  }
 },
 "share_positive_correlation": {
  "label": "Share of relations of type Positive_Correlation",
  "value": 27.0,
  "tolerance": {
   "abs": 0.5
  }
 },
 "share_negative_correlation": {
  "label": "Share of relations of type Negative_Correlation",
  "value": 17.0,
  "tolerance": {
   "abs": 0.5
  }
 },
 "mentions_per_doc": {
  "label": "Entity mentions per document",
  "value": 34.0,
  "tolerance": {
   "abs": 0.5
  }
 },
 "relations_per_doc": {
  "label": "Relations per document",
  "value": 10.8,
  "tolerance": {
   "abs": 0.05
  }
 }
}''')
PUBLISHED_FROM = 'Europe PMC full-text record PMC9487702 (XML, no page numbers): Table 4, Number of entity (mention and identifier) and relation annotations in the BioRED corpus; Methods, section Data characteristics | https://www.ebi.ac.uk/europepmc/webservices/rest/PMC9487702/fullTextXML'
DATA_DIR = pathlib.Path(os.environ.get('DATAKIT_DATA', 'datakit-data/' + LAB))


def data_file(file_id):
    """Path of one dataset file. Downloads it from its source on first use and checks the checksum."""
    f = FILES[file_id]
    path = DATA_DIR / f['name']
    if not path.exists():
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        print('downloading', f['url'])
        urllib.request.urlretrieve(f['url'], path)
    if f['sha256']:
        got = hashlib.sha256(path.read_bytes()).hexdigest()
        if got != f['sha256']:
            print('WARNING:', path.name, 'differs from the copy this lab was checked against (the source file changed)')
    return path


def within(computed, check):
    tol, pub = check['tolerance'], check['value']
    if 'exact' in tol:
        return computed == pub
    limit = tol['abs'] if 'abs' in tol else tol['rel'] * abs(pub)
    return abs(computed - pub) <= limit + 1e-12


def report(results):
    """Print computed against published values and save the computed values for the harness."""
    print('Published values:', PUBLISHED_FROM)
    print(f"{'check':<44}{'published':>12}{'computed':>14}  result")
    for key, check in PUBLISHED.items():
        got = results.get(key)
        shown = '(missing)' if got is None else (f'{got:.4f}' if isinstance(got, float) else str(got))
        verdict = 'missing' if got is None else ('match' if within(got, check) else 'DIFFERENT')
        print(f"{check['label'][:43]:<44}{str(check['value']):>12}{shown:>14}  {verdict}")
    out = pathlib.Path(os.environ.get('DATAKIT_RESULTS', 'results.json'))
    out.write_text(json.dumps(results, indent=1))
    print('saved', out)


## 1. The notice and the files

The README carries the licence notice and the citation the authors ask for. The zip holds the corpus in two formats.

In [ ]:
import io
import re
import zipfile
from collections import Counter

import pandas as pd

readme = data_file("readme").read_text(encoding="utf-8", errors="replace")
notice = readme.split("Contents:")[0]
print(notice.strip())

zf = zipfile.ZipFile(data_file("corpus"))
members = pd.DataFrame([(i.filename, i.file_size, "%04d-%02d-%02d" % i.date_time[:3]) for i in zf.infolist() if not i.is_dir()],
                       columns=["file", "bytes", "date"])
print(members.to_string(index=False))

## 2. The PubTator format

The annotation guideline says BioRED uses the PubTator format. It is plain text. Each document is a block of lines, and blocks are separated by an empty line:

- `PMID|t|title` and `PMID|a|abstract` hold the text;
- an **entity line** has six tab-separated fields: PMID, start offset, end offset, the span text, the type, the identifier;
- a **relation line** has five fields: PMID, relation type, identifier 1, identifier 2, and the novelty label (`Novel` or `No`).

A relation links two identifiers, not two spans: it says that two concepts are related somewhere in that abstract.

Pick the PubTator file of each split by its name.

In [ ]:
def split_of(name):
    low = name.lower()
    for key, split in (("train", "train"), ("dev", "dev"), ("test", "test")):
        if key in low:
            return split
    return None

pubtator = {split_of(f): f for f in members["file"] if f.lower().endswith(".pubtator") and split_of(f)}
assert sorted(pubtator) == ["dev", "test", "train"], f"expected one PubTator file per split, found {pubtator}"
for split, f in pubtator.items():
    print(split, "->", f)

In [ ]:
TEXT = re.compile(r"^(\d+)\|([ta])\|")

def parse(text, split):
    docs, mentions, relations, other = set(), [], [], []
    for line in text.splitlines():
        if not line.strip():
            continue
        m = TEXT.match(line)
        if m:
            docs.add(m.group(1))
            continue
        f = line.split("\t")
        if len(f) >= 5 and f[1].isdigit() and f[2].isdigit():
            mentions.append({"split": split, "pmid": f[0], "start": int(f[1]), "end": int(f[2]), "span": f[3],
                             "label": f[4], "identifier": f[5] if len(f) > 5 else ""})
        elif len(f) >= 4:
            relations.append({"split": split, "pmid": f[0], "type": f[1], "id1": f[2], "id2": f[3],
                              "novelty": f[4] if len(f) > 4 else ""})
        else:
            other.append(line)
    return docs, mentions, relations, other

docs, men, rel = {}, [], []
for split, f in pubtator.items():
    d, m, r, other = parse(zf.read(f).decode("utf-8"), split)
    assert not other, f"{len(other)} lines in {f} are neither text, entity nor relation lines"
    docs[split] = d
    men += m
    rel += r
mentions = pd.DataFrame(men)
relations = pd.DataFrame(rel)
print({s: len(d) for s, d in docs.items()}, "documents")
print(len(mentions), "entity lines;", len(relations), "relation lines")

Look at the annotation lines of one document. The span text is a few words cut from the abstract; the offsets say where.

In [ ]:
first = sorted(docs["train"])[0]
print(mentions[mentions.pmid == first][["start", "end", "span", "label", "identifier"]].head(12).to_string(index=False))
print(relations[relations.pmid == first][["type", "id1", "id2", "novelty"]].to_string(index=False))

## 3. From type labels to the paper's six classes

The files use longer labels than the paper. Map each label to one class by its name, and check that no label is left over or matches twice.

In [ ]:
RULES = [("Gene", ("Gene",)), ("Disease", ("Disease",)), ("Chemical", ("Chemical",)),
         ("Variant", ("Variant", "Mutation", "SNP")), ("Species", ("Organism", "Species")), ("Cell Line", ("CellLine",))]

def class_of(label):
    hits = [cls for cls, keys in RULES if any(k in label for k in keys)]
    assert len(hits) == 1, f"label {label!r} maps to {hits}"
    return hits[0]

labels = mentions["label"].value_counts()
mapping = {lab: class_of(lab) for lab in labels.index}
mentions["cls"] = mentions["label"].map(mapping)
print(pd.DataFrame({"lines": labels, "class": pd.Series(mapping)}).to_string())

## 4. Rebuild Table 4

Mentions are lines. Unique identifiers are distinct (class, identifier) pairs, counted inside each column: one split, or all 600 abstracts for Total. A composite field is split at the comma first.

In [ ]:
SPLITS = ["train", "dev", "test", "total"]
CLASSES = ["Gene", "Disease", "Chemical", "Variant", "Species", "Cell Line"]

ids = mentions.assign(identifier=mentions["identifier"].str.split(",")).explode("identifier")
ids["identifier"] = ids["identifier"].str.strip()
linked = ids[(ids["identifier"] != "") & (ids["identifier"] != "-")]

def in_split(df, split):
    return df if split == "total" else df[df["split"] == split]

table = {}
for split in SPLITS:
    col = {}
    for cls in CLASSES:
        n_mentions = int((in_split(mentions, split)["cls"] == cls).sum())
        n_ids = in_split(linked, split).query("cls == @cls")["identifier"].nunique()
        col[cls] = (n_mentions, int(n_ids))
    col["All"] = (sum(v[0] for v in col.values()), sum(v[1] for v in col.values()))
    table[split] = col
table4 = pd.DataFrame({s: {c: f"{v[0]} ({v[1]})" for c, v in table[s].items()} for s in SPLITS}).loc[["All"] + CLASSES]
table4.loc["Document"] = [len(docs[s]) if s != "total" else sum(len(d) for d in docs.values()) for s in SPLITS]
print(table4.to_string())

In [ ]:
n_rel = {s: len(in_split(relations, s)) for s in SPLITS}
n_novel = {s: int((in_split(relations, s)["novelty"] == "Novel").sum()) for s in SPLITS}
print("Relations:", n_rel)
print("Novel relations:", n_novel)
print("Novelty labels used:", dict(relations["novelty"].value_counts()))

## 5. Relation types and concept pairs

Each relation names two identifiers. Look up the class of each identifier among the mentions of the same abstract to see which concept pair it joins.

In [ ]:
types = relations["type"].value_counts()
share = (100 * types / len(relations)).round(1)
print(pd.DataFrame({"relations": types, "percent": share}).to_string())

cls_in_doc = linked.groupby(["pmid", "identifier"])["cls"].agg(lambda s: "/".join(sorted(set(s))))
key = lambda p, i: cls_in_doc.get((p, i), "not found")
relations["pair"] = [" - ".join(sorted([key(p, a), key(p, b)])) for p, a, b in zip(relations.pmid, relations.id1, relations.id2)]
print(relations["pair"].value_counts().to_string())

per_doc_mentions = len(mentions) / 600
per_doc_relations = len(relations) / 600
print(f"Per document: {per_doc_mentions:.2f} mentions, {per_doc_relations:.2f} relations")

## 6. Choices that change a count (shown, not checked)

Each line below is a place where a different rule gives a different number. None of them is a check; they tell you what to look at first if your count and a published one differ.

In [ ]:
composite = mentions["identifier"].str.contains(",")
print("Spans with more than one identifier:", int(composite.sum()))
print("Identifiers if composite fields are NOT split:",
      mentions[(mentions.identifier != "") & (mentions.identifier != "-")].groupby("cls")["identifier"].nunique().sum())
print("Spans whose identifier is '-' or empty:", int(mentions["identifier"].isin(["-", ""]).sum()))
print("Identical entity lines repeated in a document:", int(mentions.duplicated(["pmid", "start", "end", "label", "identifier"]).sum()))
print("Identical relation lines repeated in a document:", int(relations.duplicated(["pmid", "type", "id1", "id2"]).sum()))
multi = linked.groupby("identifier")["cls"].nunique()
print("Identifiers that occur in more than one class:", int((multi > 1).sum()))
split_sum = sum(table[s]["All"][1] for s in ["train", "dev", "test"])
print(f"Unique identifiers: splits add to {split_sum}, the Total column holds {table['total']['All'][1]}")
print(f"Novel share: {100 * n_novel['total'] / n_rel['total']:.1f} percent")

In [ ]:
RESULTS = {}
for s in SPLITS:
    RESULTS[f"docs_{s}"] = len(docs[s]) if s != "total" else sum(len(d) for d in docs.values())
    for cls, cid in [("All", "all"), ("Gene", "gene"), ("Disease", "disease"), ("Chemical", "chemical"),
                     ("Variant", "variant"), ("Species", "species"), ("Cell Line", "cellline")]:
        RESULTS[f"mentions_{cid}_{s}"] = table[s][cls][0]
        RESULTS[f"ids_{cid}_{s}"] = table[s][cls][1]
    RESULTS[f"relations_{s}"] = n_rel[s]
    RESULTS[f"novel_{s}"] = n_novel[s]
RESULTS["share_association"] = float(100 * types.get("Association", 0) / len(relations))
RESULTS["share_positive_correlation"] = float(100 * types.get("Positive_Correlation", 0) / len(relations))
RESULTS["share_negative_correlation"] = float(100 * types.get("Negative_Correlation", 0) / len(relations))
RESULTS["mentions_per_doc"] = per_doc_mentions
RESULTS["relations_per_doc"] = per_doc_relations
report(RESULTS)

## What the result says

Put your Table 4 next to the paper's. Where every cell agrees, you know exactly what one mention, one identifier and one relation mean in these files. Where a cell differs, go back to section 6 and ask which rule the authors may have used instead. Do not change your rule until it matches: write down the difference and its likely cause.

**This lab does not fully reproduce.** When it was run on 12 Nov 2026, 72 of its 73 checks matched the paper: every count in Table 4 (documents, mentions and unique identifiers by class and split, relations, novel relations), the shares of Association and Positive_Correlation, and the two per-document averages. The one that did not match is the share of Negative_Correlation: the files give 17.7 percent, the paper's text prints 17. The paper also prints 69 percent novel where its own Table 4 counts give 69.7, so its text may cut decimals instead of rounding. That is a likely cause, not a confirmed one; the lab keeps the half-digit tolerance and reports the difference.

## Try next

1. Count unique identifiers without splitting composite fields. Which class changes most, and why?
2. Pick one relation type, for example `Bind`. Which concept pairs does it join? Check the guideline's table of pairs and types.
3. Find the identifier that occurs in more than one class. Open its abstract on PubMed and decide whether the class or the identifier is the problem.
4. The paper says 2 percent of relations "are involved in the triple relations". Which relation types in your table could that mean? Can you get 2 from any of them?